# Exploring the SMHI data

Before writing `ingest_data.py` I wanted to look at the raw file first. SMHI CSVs are not standard CSVs and a naive `pd.read_csv(url)` will completely misparse it. This notebook shows what I found and why I made the parsing decisions I did.

This was a one-time exploration. The specific temperatures and station values shown here reflect a single snapshot - the structure is what matters, not the numbers.

In [11]:
import requests
import pandas as pd
from io import StringIO

SMHI_URL = (
    "https://opendata-download-metobs.smhi.se"
    "/api/version/1.0/parameter/1/station-set/all/period/latest-hour/data.csv"
)

response = requests.get(SMHI_URL)

First I just print the raw lines to see the actual structure.

In [12]:
raw_lines = response.text.splitlines()

for i, line in enumerate(raw_lines[:12]):
    print(f"[{i}] {line}")

[0] ﻿Parameternamn;Beskrivning;Enhet
[1] Lufttemperatur;momentanvärde, 1 gång/tim;celsius
[2] 
[3] StationsId;Stationsnamn;Latitude;Longitude;Height;2026-06-09 11:00:00;Kvalitet;;Data från senaste timmen
[4] 188790;Abisko Aut;68.3538;18.8164;392.235;19.4;G;;Tidsperiod (fr.o.m.) = 2026-06-09 10:00:01 (UTC)
[5] 97280;Adelsö A;59.3579;17.5213;5.612;19.6;G;;Tidsperiod (t.o.m.) = 2026-06-09 11:00:00 (UTC)
[6] 9013;Alskog;57.3271;18.6162;0.0;16.8;Y;;Samplingstid = Ej angivet
[7] 167710;Arjeplog A;66.0513;17.8396;430.839;22.5;G;;
[8] 159880;Arvidsjaur A;65.5941;19.2642;382.45;20.3;G;;Kvalitetskoderna:
[9] 92410;Arvika A;59.6743;12.6354;65.758;19.1;G;;Grön (G) = Kontrollerade och godkända värden.
[10] 98040;Berga;59.068;18.115;4.3;15.6;G;;Gul (Y) = Misstänkta eller aggregerade värden. Grovt kontrollerade arkivdata och okontrollerade realtidsdata (senaste 2 tim).
[11] 151280;Bjuröklubb A;64.4799;21.5754;35.41;16.2;G;;


Lines 0-2 are parameter metadata, not data. The actual header is on line 3, and data starts at line 4. That is why I use `skiprows=3` and `sep=";"`.

The other thing I noticed on line 3 is that column 5 has a timestamp as its name, like `2026-06-09 11:00:00`. That changes every hour. The name of the column is the observation time, and the values inside it are the temperature readings.

The file also starts with a UTF-8 BOM. If I don't handle it the first column name gets a garbage prefix.

In [13]:
print("First bytes:", response.content[:3].hex())
print("BOM present:", response.content[:3] == b'\xef\xbb\xbf')

First bytes: efbbbf
BOM present: True


Now I can read it correctly with `skiprows=3`, `sep=";"`, and `encoding="utf-8-sig"`.

In [14]:
df = pd.read_csv(StringIO(response.text), sep=";", skiprows=3, encoding_errors="replace")

print("Shape:", df.shape)
print()
for i, col in enumerate(df.columns):
    print(f"  [{i}] {repr(col)}")

Shape: (184, 9)

  [0] 'StationsId'
  [1] 'Stationsnamn'
  [2] 'Latitude'
  [3] 'Longitude'
  [4] 'Height'
  [5] '2026-06-09 11:00:00'
  [6] 'Kvalitet'
  [7] 'Unnamed: 7'
  [8] 'Data från senaste timmen'


Columns 7 and 8 are annotation text that SMHI embeds in the CSV, not actual data fields. I'll drop them.

In [15]:
df.iloc[:6, 7:]

,Unnamed: 7,Data från senaste timmen
0,NaN,Tidsperiod (fr.o.m.) = 2026-06-09 10:00:01 (UTC)
1,NaN,Tidsperiod (t.o.m.) = 2026-06-09 11:00:00 (UTC)
2,NaN,Samplingstid = Ej angivet
3,NaN,NaN
4,NaN,Kvalitetskoderna:
5,NaN,Grön (G) = Kontrollerade och godkända värden.


Keep columns 0-6 and grab the observation timestamp from column 5 before renaming.

In [16]:
obs_time = df.columns[5]
print("Observation time:", obs_time)

df = df.iloc[:, :7].copy()

Observation time: 2026-06-09 11:00:00


In [17]:
df.columns = ["station_id", "station_name", "latitude", "longitude", "height_m", "temperature_c", "quality_code"]
df["observation_time"] = pd.to_datetime(obs_time)

In [18]:
df["temperature_c"] = pd.to_numeric(df["temperature_c"], errors="coerce")
df["latitude"]      = pd.to_numeric(df["latitude"],      errors="coerce")
df["longitude"]     = pd.to_numeric(df["longitude"],     errors="coerce")
df["height_m"]      = pd.to_numeric(df["height_m"],      errors="coerce")

df = df.dropna(subset=["temperature_c"])
print(f"{len(df)} stations with readings")

184 stations with readings


In [19]:
df.head(10)

,station_id,station_name,latitude,longitude,height_m,temperature_c,quality_code,observation_time
0,188790,Abisko Aut,68.3538,18.8164,392.235,19.4,G,2026-06-09 11:00:00
1,97280,Adelsö A,59.3579,17.5213,5.612,19.6,G,2026-06-09 11:00:00
2,9013,Alskog,57.3271,18.6162,0.000,16.8,Y,2026-06-09 11:00:00
3,167710,Arjeplog A,66.0513,17.8396,430.839,22.5,G,2026-06-09 11:00:00
4,159880,Arvidsjaur A,65.5941,19.2642,382.450,20.3,G,2026-06-09 11:00:00
5,92410,Arvika A,59.6743,12.6354,65.758,19.1,G,2026-06-09 11:00:00
6,98040,Berga,59.0680,18.1150,4.300,15.6,G,2026-06-09 11:00:00
7,151280,Bjuröklubb A,64.4799,21.5754,35.410,16.2,G,2026-06-09 11:00:00
8,92130,Blomskog A,59.2213,12.0754,169.645,14.5,G,2026-06-09 11:00:00
9,132110,Blåhammaren A,63.1854,12.1734,1086.952,12.2,G,2026-06-09 11:00:00


Quick quality check before loading to Postgres.

In [20]:
print("Nulls per column:")
print(df.isnull().sum())

Nulls per column:
station_id          0
station_name        0
latitude            0
longitude           0
height_m            0
temperature_c       0
quality_code        0
observation_time    0
dtype: int64


In [21]:
# G = verified and approved, Y = suspect or unverified
print(df["quality_code"].value_counts())

quality_code
G    178
Y      6
Name: count, dtype: int64


In [22]:
print(df["temperature_c"].describe().round(2))
print()
print("Coldest:", df.loc[df["temperature_c"].idxmin(), "station_name"], df["temperature_c"].min(), "C")
print("Warmest:", df.loc[df["temperature_c"].idxmax(), "station_name"], df["temperature_c"].max(), "C")

count    184.00
mean      17.86
std        2.79
min       10.10
25%       15.60
50%       18.10
75%       20.05
max       23.40
Name: temperature_c, dtype: float64

Coldest: Tarfala A 10.1 C
Warmest: Kvikkjokk-Årrenjarka A 23.4 C
